# FASE03 Recap

**ROOT:** `/Users/emilevandevoorde/Documents/mechelen_parking/`  
**DATA_PROC:** `ROOT/data_processed/`

**Twee administratieve tiers (tier_admin):**
- `centrum` → P Grote Markt, P Hoogstraat, P Kathedraal, P Lamot, P Veemarkt  
- `vesten_of_rand` → P Bruul, P Keerdok, P Komet, P Maarten, P Tinel

---

### Data status
| Bestand | Rijen | Kolommen | Periode |
|---|---|---|---|
| `train_features.parquet` | 129.837 | 40 | 2020, 2023, 2024 |
| `holdout_features.parquet` | 87.600 | 40 | 2025 |

**Effectieve train na lag-drop (nb09): ~108.519 rijen (83.6%)**  
→ Drop alle rijen waar `occ_lag_168h` IS NULL vóór modeltraining

**Target:** `occupancy_rate` (continue, [0,1])

---

### Finale feature-matrix (35 features → 40 na one-hot)

| Categorie | Features |
|---|---|
| Temporeel cyclisch (6) | `hour_sin/cos`, `weekday_sin/cos`, `month_sin/cos` |
| Kalender (5, +2 OH) | `day_type_3`*, `is_school_vacation`, `is_national_holiday`, `is_any_holiday`, `year_2020` |
| Autoregressive (3) | `occ_lag_1h`, `occ_lag_24h`, `occ_lag_168h` |
| Weer (3, +3 OH) | `precip_bin`*, `wind_strong`, `sun_duration_scaled` |
| Ruimtelijk (5) | `mean_occ_by_parking`, `behavioral_cluster`, `tier_num`, `high_lt_pressure`, `log_capacity` |
| Events (6) | `is_football_day`, `is_festival_day`, `is_kermis_day`, `is_carnival_day`, `is_procession_day`, `n_concurrent_events` |
| Interacties (5) | `voetbal/festival/carnaval/vakantie/feestdag_x_centrum` |
| Cascade (2) | `hours_since_last_event`, `hours_to_next_event` |

*one-hot in nb09 met `drop_first=True`  
**Metadata (niet als modelfeature):** `rounded_hour`, `parking_id`, `tier_admin`, `year`, `occupancy_rate`

---

### Kritische nb08-bevindingen, van belang voor nb09

1. **Lag-NaN via tijdsbewuste self-join** — shift() was fout door tijdreeksgaten (gem. 29% uitval per parking na kwaliteitsfilter nb05). Oplossing geïmplementeerd. Drop rijen waar `occ_lag_168h` NaN is.

2. **VIF `mean_occ_by_parking` (10.97) en `behavioral_cluster` (11.10)** — structureel gecorreleerd maar aanvaardbaar voor boomgebaseerde modellen. Beide behouden.

3. **`wind_strong` spaarzaam (0.2%)** — evalueer via permutatie-importance in nb11; drop indien verwaarloosbaar.

4. **`year_2020` = 0 voor volledige holdout** — correct, 2025 is out-of-sample. Niet ordinaal behandelen.

5. **`precip_bin` verdeling:** droog=84.6%, licht=14.1%, matig=1.0%, zwaar=0.2%

---

### Nb09 — wat moet gebeuren (zie ook 'Plan van Aanpak en Beslissingen')
1. Laden `train_features.parquet` + `holdout_features.parquet`
2. Drop lag-NaN rijen (`occ_lag_168h` als conservatieve mask)
3. One-hot encoding `precip_bin` (ref=droog) + `day_type_3` (ref=weekday)
4. Train/validatie split — **tijdsbewust** (geen random split!)
5. Baseline model: naïeve benchmark (gemiddelde per uur×parking×dag_type)
6. Eerste modeltraining: Random Forest + XGBoost, tier-gestratificeerd
7. Evaluatie: RMSE, MAE, R² — gesplitst per tier


# Nb09 — Plan van Aanpak: Baseline Modellering

## Probleemstelling en doelstelling

Het centrale modelleerdoel van deze thesis is **uurlijkse bezettingsgraad 
voorspellen per parking**, met expliciete tier-stratificatie, om 
scenario-gebaseerde beleidsimulaties mogelijk te maken (nb13). Nb09 
legt daarvoor het fundament: een robuuste, goed gedocumenteerde 
baseline die als ijkpunt dient voor alle volgende modelvergelijkingen.

De kwaliteit van de baseline bepaalt de interpretatie van het 
modelresultaat: een zwakke baseline maakt elk model "goed"; een 
sterke baseline dwingt tot een echt informatief model.

---

## Beslissing 1 — Validatiestrategie

### Waarom niet: random split

Random train/test-splits zijn **categorisch incorrect** voor 
tijdreeksdata. Door de sterke temporele autocorrelatie in 
bezettingsdata (ACF-pieken op lag 1, 24 en 168 uur, H-T2) 
lekt toekomstige informatie achterwaarts in de trainingsset, 
wat de validatieprestatie systematisch overschat (Roberts 
et al., 2017; Cerqueira et al., 2023).

### Waarom niet: walk-forward rolling-window CV

Walk-forward validatie hertraint het model na elke stap. Dit 
is de gouden standaard voor echte one-step-ahead forecasting 
(Tashman, 2000). In onze setting is het echter overbodig: het 
model wordt éénmalig getraind op historische data en 
geëvalueerd op een volledig apart toekomstig jaar (2025). 
Walk-forward CV zou bovendien met de bestaande tijdreeksgaten 
complex te implementeren zijn zonder extra leakage-risico.

### Keuze: **temporeel geblokte validatie — 2024 als validatiejaar**

De trainingsdata (2020, 2023, 2024) wordt gesplitst in:

| Set | Jaren | N rijen (na lag-drop) |
|---|---|---|
| Treinset | 2020 + 2023 | ~75.000 (schatting) |
| Validatieset | 2024 | ~33.000 (schatting) |
| Holdout (nooit aanraken) | 2025 | ~85.920 |

**Motivatie:** 2024 is het meest recente jaar in de trainingsdata, 
ligt structureel het dichtst bij de holdout (2025), en bevat 
volledige jaardekking voor alle 10 parkings. Dit simuleert de 
echte inzetomgeving: train op verleden, evalueer op meest 
recent beschikbare jaar (Cerqueira et al., 2023). Blokken per 
volledig jaar elimineert temporele afhankelijkheid tussen 
train- en validatierijen volledig (Roberts et al., 2017).

> *"Temporele blokkering per kalenderjaar wordt gebruikt als 
> validatiestrategie, waarbij 2024 fungeert als het 
> interne validatiejaar. Deze aanpak simuleert de 
> productie-inzetomgeving en voorkomt temporele 
> dataleakage (Roberts et al., 2017; Cerqueira et al., 2023)."*

---

## Beslissing 2 — Baseline (ijkpunt)

### Keuze: **seizoensnaïeve baseline (sNaive)**

De ijkpuntbaseline is de **seizoensnaïeve voorspeller**: voor elke 
observatie op tijdstip t wordt de bezettingsgraad van exact 
één week geleden (t − 168 uur) als voorspelling gebruikt — 
identiek aan `occ_lag_168h`. Dit is de standaard benchmark 
in parkeer- en verkeersprognoseliteratuur (Lira et al., 2021; 
Vlahogianni et al., 2014).

Een tweede, rijkere baseline is de **historisch gemiddelde 
bezettingsgraad per (parking_id × hour × day_type_3)** berekend 
op de treinset. Dit is een veelgebruikte praktijkbenchmark 
bij parkeerbeheerders.

| Baseline | Formule | Motivatie |
|---|---|---|
| sNaive | ŷ = occ(t−168h) | Standaard tijdreeksbenchmark |
| HistMean | ŷ = μ(parking, hour, day_type_3) | Praktijkbenchmark parkingbeheer |

Elk ML-model dat deze baselines niet significant verslaat, 
heeft geen toegevoegde waarde boven heuristisch beheer.

---

## Beslissing 3 — Modelkeuze

### Keuze: **XGBoost als primair model, Random Forest als secundair**

**XGBoost** (Chen & Guestrin, 2016) is het primaire model op basis 
van drie argumenten:

1. **Empirische superioriteit op tabulaire data** — XGBoost won 
   elke KDD Cup en Kaggle-competitie op gestructureerde 
   tabulaire data in de periode 2014–2019 (Chen & Guestrin, 2016). 
   De combinatie van gradient boosting, L1/L2-regularisatie en 
   sparse-aware split-finding maakt het robuust voor de 
   spaarzame eventfeatures (0.1–1.7% positief).

2. **Expliciete regularisatie** — De `reg_alpha` (L1) en 
   `reg_lambda` (L2) parameters voorkomen overfitting op de 
   spaarzame eventdummies en de hoge multicollineariteit 
   tussen `mean_occ_by_parking` en `behavioral_cluster`.

3. **Compatibiliteit met SHAP** — XGBoost integreert native 
   met SHAP-waarden (Lundberg & Lee, 2017) voor nb12, wat 
   vereist is voor de beleidsinterpretatie in nb13.

**Random Forest** (Breiman, 2001) wordt als secundair model getraind 
als stabiliteitsvergelijking: hoge variantiereductie via bagging 
maakt het minder gevoelig voor uitschieters in de 
trainingsset (w.o. COVID-2020 periode). Het verschil in 
prestatie tussen XGBoost en Random Forest is ook 
informatief: een groot verschil suggereert dat sequentiële 
foutcorrectie (boosting) wezenlijk bijdraagt.

**Lineaire regressie** (ElasticNet) wordt als tertiaire baseline 
toegevoegd — niet als productiemodel, maar als 
interpretabiliteitsbenchmark: als XGBoost amper beter 
presteert dan ElasticNet, zijn de niet-lineaire patronen 
in de data beperkt.

| Model | Type | Primaire rol |
|---|---|---|
| sNaive / HistMean | Heuristisch | Ijkpunt |
| ElasticNet | Lineair | Lineariteitsbenchmark |
| Random Forest | Ensemble bagging | Stabiliteitscontrole |
| **XGBoost** | **Ensemble boosting** | **Primair productiemodel** |

---

## Beslissing 4 — Tier-stratificatie: één model vs. twee modellen

### Keuze: **twee afzonderlijke modellen per tier**

De thesis-titel luidt *"Tier-Stratified Occupancy Prediction"* — 
dit is geen decoratieve toevoeging maar een methodologische 
kernkeuze. De FASE02-analyses toonden fundamenteel 
verschillende gedragspatronen per tier:

- Voetbal: **negatief** effect in centrum, **positief** in vesten
- Festival: **positief** effect in centrum, geen effect in vesten
- Schoolvakantie: sterkste effect in centrum, beperkt in vesten

Een globaal model met `tier_num` en interactietermen kan deze 
heterogeniteit partieel vangen, maar dwingt het model om een 
gedeeld functie-approximatie te zoeken over twee structureel 
verschillende populaties. Aparte modellen per tier laten elk 
model volledig vrij om tier-specifieke non-lineaire patronen 
te leren (Hastie et al., 2009).

**Praktische consequentie voor nb13:** tier-gestratificeerde 
modellen produceren tier-specifieke scenario-uitkomsten, 
wat de beleidsrelevantie vergroot.

> *"Twee afzonderlijke XGBoost-modellen worden getraind — 
> één per administratieve tier — op basis van de empirisch 
> vastgestelde structureel heterogene gedragspatronen 
> (FASE02, nb05–nb07). Een globaal model met 
> interactietermen zou deze heterogeniteit partieel 
> maskeren en de tier-specifieke beleidsinterpretatie 
> bemoeilijken (Hastie et al., 2009)."*

---

## Beslissing 5 — Evaluatiemetrieken

### Keuze: **RMSE als primaire metriek, MAE en R² als secundair**

| Metriek | Formule | Motivatie |
|---|---|---|
| **RMSE** | √(Σ(ŷ−y)²/n) | Penaliseert grote fouten zwaarder — relevant voor capaciteitsbeheer waar piekfouten kostbaar zijn |
| MAE | Σ\|ŷ−y\|/n | Robuust voor uitschieters — geeft mediaan foutgrootte |
| R² | 1 − SS_res/SS_tot | Proportie verklaarde variantie — vergelijkbaar over tiers |

**MAPE wordt niet gebruikt:** bij lage bezettingsgraden (P Maarten, 
gem. 6.4%) produceert MAPE instabiele waarden door deling 
door near-zero observaties.

Evaluatie geschiedt op **drie niveaus**:
1. **Globaal** — volledige validatieset en holdout
2. **Per tier** — centrum vs. vesten_of_rand
3. **Per parking** — om parking-specifieke modelfailures te detecteren

---

## Beslissing 6 — Hyperparameter tuning

### Keuze: **Bayesiaanse optimalisatie met temporeel geblokte CV**

Standaard `GridSearchCV` of `RandomizedSearchCV` met willekeurige 
k-fold splits zijn incorrect voor tijdreeksdata (Roberts et al., 
2017). De hyperparameter-tuning gebruikt `TimeSeriesSplit` 
(n_splits=3) zodat elke tuning-fold de temporele ordening 
respecteert. Bayesiaanse optimalisatie via `optuna` is 
efficiënter dan grid search voor het XGBoost-hyperparameter-
ruimte (learning_rate, max_depth, n_estimators, subsample, 
colsample_bytree, reg_alpha, reg_lambda).

---
